# Preliminaries

## Imports

In [ ]:
from pathlib import Path

import holoviews as hv
import panel as pn

hv.extension("bokeh")
pn.extension()
# If in google colab, run hack that allows holoviews to work properly
try:
    import google.colab  # noqa

    def _render(self, **kwargs):
        hv.extension("bokeh")
        return hv.Store.render(self)

    hv.core.Dimensioned._repr_mimebundle_ = _render
except ModuleNotFoundError:
    pass

TMP_NOTEBOOK_ROOT = Path("/tmp/bridge-ds/tutorials")

## Loading a dataset

To create BridgeDS Dataset objects, it's recommended to utilize a **DatasetProvider**. In this instance, we'll employ the Coco2017Detection provider:

In [ ]:
from bridge.providers.vision import Coco2017Detection

root_dir = TMP_NOTEBOOK_ROOT / "coco"

provider = Coco2017Detection(root_dir, split="train", img_source="stream")
ds = provider.build_dataset()
ds

# TableAPI

In BridgeDS, we use two complementing approaches to view datasets. We call them the **Sample API** and the **Table API**. This tutorial is about the latter.

The Table API can be described as:

    A dataset can be viewed as a table where every row represents a single element. Elements have a unique element_id but share their sample_id with other elements from the same Sample. The element_id and sample_id columns serve as the table's multi-index.

## The elements table
All elements in a Dataset live in a single Pandas DataFrame, exposed as `ds.elements`. Each row is one element; rows are distinguished by their `role` column. For COCO, the two roles are `"image"` (one per sample) and `"bbox"` (zero or more per sample). To get a "samples" / "annotations" view, filter by role:

In [ ]:
from bridge.utils.constants import ELEMENT_COLS

ds.elements[ds.elements[ELEMENT_COLS.ROLE] == "image"].head()

In [ ]:
ds.elements[ds.elements[ELEMENT_COLS.ROLE] == "bbox"].head()

## Methods
The Table API is designed to expose callables that accept a single Pandas DataFrame (the full `ds.elements` table) and return a result that fits Pandas indexing. The following sections showcase methods that allow users to perform different actions on Datasets.

### Filter
We can filter the elements with `ds.select`, using familiar Pandas syntax. The selector callback receives the full elements DataFrame; the return value can be a boolean mask, an index, or an array of sample IDs:

In [ ]:
def keep_samples_with_license_lt_3(elements):
    image_rows = elements[elements[ELEMENT_COLS.ROLE] == "image"]
    keep_sample_ids = image_rows[image_rows.license < 3].index.get_level_values("sample_id")
    return elements.index.get_level_values("sample_id").isin(keep_sample_ids)


def keep_iscrowd_bboxes(elements):
    # keep all non-bbox rows; among bbox rows, keep only iscrowd != 0
    is_bbox = elements[ELEMENT_COLS.ROLE] == "bbox"
    return ~is_bbox | (elements.iscrowd != 0)


def drop_samples_without_bboxes(elements):
    bbox_sample_ids = elements[elements[ELEMENT_COLS.ROLE] == "bbox"].index.get_level_values("sample_id").unique()
    return elements.index.get_level_values("sample_id").isin(bbox_sample_ids)


print("Original dataset:")
print(ds, "\n")
print("Keep images (and corresponding bboxes) where the license < 3:")
print(ds.select(keep_samples_with_license_lt_3), "\n")
print("Keep only bboxes with iscrowd != 0. This leaves us with some empty images:")
print(ds.select(keep_iscrowd_bboxes), "\n")
print("We can chain selectors to filter the bboxes, and subsequently drop empty images:")
print(ds.select(keep_iscrowd_bboxes).select(drop_samples_without_bboxes))

### Assign
We can assign new columns to the elements table using `ds.assign`, with familiar Pandas-style syntax. Let's add an `n_bboxes` column counting how many bboxes each sample has, broadcast across every row in that sample (so both image and bbox rows get the same value):

In [ ]:
def n_bboxes_per_sample(elements):
    # How many bbox rows per sample_id, then broadcast to every row in the sample.
    bbox_counts = (
        elements[elements[ELEMENT_COLS.ROLE] == "bbox"].groupby("sample_id").size()
    )
    return elements.index.get_level_values("sample_id").map(bbox_counts).fillna(0).astype(int)


ds = ds.assign(n_bboxes=n_bboxes_per_sample)
ds.elements[ds.elements[ELEMENT_COLS.ROLE] == "image"].head()

### Sorting
We can sort the tables using familiar Pandas syntax:

In [ ]:
sorted_ds = ds.sort("n_bboxes", ascending=False)
sorted_ds.elements[sorted_ds.elements[ELEMENT_COLS.ROLE] == "image"].head()

Note that if we sort the samples table, we can change the positional index used by the Sample API (ds.iget) which dictates the order of the samples below. The next cell will show the dataset in order from most bboxes per image to least:

In [ ]:
sorted_ds.show()